# 15. Best model
Nhánh GitHub: `15-best-model`

Chọn mô hình tốt nhất theo Macro-F1 validation (từ notebook 14), chọn ngưỡng "Không xác định" bằng validation, huấn luyện lại trên toàn bộ 2.000 clip và xuất file cho `app.py`. Mô hình toàn bộ dữ liệu không có tập test riêng nên **không dùng để báo cáo độ chính xác** (báo cáo lấy kết quả kiểm định chéo).

In [1]:
!pip -q install soundfile
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/drive/MyDrive/ESC50')
from esc_lib import *
setup()

Mounted at /content/drive
GPU: Tesla T4 | VRAM 15.6 GB | torch 2.14.1+cu130 | CUDA 13.0
RAM trống 11.7 GB | ổ đĩa trống 120.8 GB
Mixup: BẬT | quick: False | kết quả lưu ở: /content/drive/MyDrive/ESC50/results_mixup


In [2]:
meta, X, X_DEV, y, folds = load_data()

  0%|          | 0/2000 [00:00<?, ?it/s]

X: (2000, 110250) | y: (2000,) | nằm trên: cuda:0


## Chọn best model

In [3]:
choice = json.loads((RESULT_DIR / "ensemble_choice.json").read_text())
val_best, best_fold, members = choice["val_best"], choice["best_fold"], choice["members"]
runs, rob = load_results()
FOLDS_DONE = sorted(runs["fold"].unique())
LR_BEST = {p.stem.replace("best_lr_", ""): json.loads(p.read_text())["lr"] for p in RESULT_DIR.glob("best_lr_E*.json")}

cv = runs.groupby("exp")[["acc", "macro_f1", "val_f1"]].mean().mul(100).round(2)
display(cv.sort_values("val_f1", ascending=False))
print(f"Best model (theo Macro-F1 validation): {val_best} | fold tốt nhất: {best_fold}")
print(f"Kiểm định chéo: Accuracy {cv.loc[val_best, 'acc']:.2f}% | Macro-F1 {cv.loc[val_best, 'macro_f1']:.2f}%")

shutil.copy(CKPT_DIR / f"{val_best}_fold{best_fold}.pt", BEST_DIR / f"best_{val_best}_fold{best_fold}.pt")
info = dict(experiment=val_best, config=EXPERIMENTS[val_best], fold=int(best_fold), members=members,
            cv_acc=float(cv.loc[val_best, "acc"]), cv_macro_f1=float(cv.loc[val_best, "macro_f1"]),
            tta_shifts=list(TTA_SHIFTS), mixup=USE_MIXUP)
(BEST_DIR / "best_model_info.json").write_text(json.dumps(info, indent=2, ensure_ascii=False))

,acc,macro_f1,val_f1
exp,,,
E9,93.80,93.57,94.68
E3,83.15,82.45,84.37
E4,82.90,82.34,84.27
E2,84.55,83.95,84.08
E1,80.80,80.06,81.97
E7,79.45,78.68,79.80
E8,78.10,77.23,79.66
E6,77.25,76.16,77.89
E5,75.65,74.59,77.04


Best model (theo Macro-F1 validation): E9 | fold tốt nhất: 1
Kiểm định chéo: Accuracy 93.80% | Macro-F1 93.57%


271

## Chọn ngưỡng "Không xác định" bằng validation
Ngưỡng nhỏ nhất để các clip có xác suất cao nhất ≥ ngưỡng đạt độ chính xác ≥ 95% trên các fold validation (có TTA).

In [4]:
TARGET_ACC = 0.95
thr_members = members
conf, correct = [], []
for fold in FOLDS_DONE:
    va_idx = get_split(fold)[1]
    parts = []
    for exp in thr_members:
        model, front, mean, std, src = load_member(exp, fold)
        p, y_va = predict_proba(model, front, mean, std, src, va_idx, shifts=TTA_SHIFTS)
        parts.append(p)
        del model, front
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    prob = sum(parts) / len(parts)
    conf.append(prob.max(axis=1))
    correct.append(prob.argmax(axis=1) == y_va)
conf, correct = np.concatenate(conf), np.concatenate(correct)

rows_t = []
for th in np.round(np.arange(0.05, 0.96, 0.05), 2):
    keep = conf >= th
    rows_t.append({"Ngưỡng": float(th),
                   "Tỷ lệ clip được chấp nhận (%)": round(100 * keep.mean(), 1),
                   "Độ chính xác phần được chấp nhận (%)": round(100 * correct[keep].mean(), 1) if keep.any() else np.nan})
thr_table = pd.DataFrame(rows_t)
display(thr_table)

ok = thr_table[thr_table["Độ chính xác phần được chấp nhận (%)"] >= TARGET_ACC * 100]
VAL_THRESHOLD = float(ok["Ngưỡng"].min()) if len(ok) else 0.5
print(f"Độ chính xác chung trên validation (không loại clip nào): {100 * correct.mean():.1f}% ({len(correct)} clip)")
print(f"Ngưỡng được chọn: {VAL_THRESHOLD:.2f}"
      + ("" if len(ok) else " (không ngưỡng nào đạt mục tiêu, dùng 0,5 làm mặc định)"))

Đọc 32 kHz:   0%|          | 0/2000 [00:00<?, ?it/s]

,Ngưỡng,Tỷ lệ clip được chấp nhận (%),Độ chính xác phần được chấp nhận (%)
0,0.05,100.0,94.6
1,0.10,100.0,94.6
2,0.15,99.0,95.1
3,0.20,98.0,95.7
4,0.25,96.7,96.1
5,0.30,95.2,96.4
6,0.35,92.9,96.9
7,0.40,90.4,97.3
8,0.45,87.9,97.9
9,0.50,85.0,98.1


Độ chính xác chung trên validation (không loại clip nào): 94.5% (2000 clip)
Ngưỡng được chọn: 0.15


## Huấn luyện mô hình cuối trên toàn bộ dữ liệu và xuất file cho app.py
Giữ siêu tham số như lúc kiểm định chéo, lấy mô hình ở epoch cuối. E9 chạy 20 epoch.

In [5]:
RUN_FINAL_FULL = True
FINAL_EXPS = None
FINAL_THRESHOLD = VAL_THRESHOLD
FINAL_PANNS_EPOCHS = 20


def final_path(exp):
    return FINAL_DIR / (f"{exp}_full_ep{FINAL_PANNS_EPOCHS}.pt" if exp == "E9" else f"{exp}_full.pt")


if RUN_FINAL_FULL:
    final_exps = list(FINAL_EXPS) if FINAL_EXPS else members
    print("Các mô hình cuối:", final_exps)
    if "E9" in final_exps:
        load_panns()
    all_idx = np.arange(len(X))
    for exp in final_exps:
        path = final_path(exp)
        if path.exists():
            print(f"{exp}: đã có {path.name}, bỏ qua")
            continue
        set_seed(CFG["SEED"])
        t0 = time.time()
        if exp == "E9":
            out = train_panns(all_idx, None, {**PANNS_CFG, "EPOCHS": FINAL_PANNS_EPOCHS, "PATIENCE": FINAL_PANNS_EPOCHS},
                              desc="E9 | toàn bộ dữ liệu")
            model, frontend, mean, std, src = out["model"], out["frontend"], 0.0, 1.0, get_x32()
            ck = dict(model="panns", n_mels=64, conv_per_block=2, sample_rate=PANNS_CFG["SAMPLE_RATE"],
                      state={k: (v.cpu().half() if v.is_floating_point() else v.cpu())
                             for k, v in model.state_dict().items()}, mean=0.0, std=1.0)
        else:
            e = EXPERIMENTS[exp]
            cfg = {**CFG, "LR": LR_BEST.get(exp, CFG["LR"])}
            out = train_model(e["model"], e["n_mels"], e["specaug"], all_idx, None, cfg,
                              verbose=False, desc=f"{exp} | toàn bộ dữ liệu")
            model, frontend, mean, std, src = out["model"], out["frontend"], out["mean"], out["std"], X_DEV
            ck = dict(model=e["model"], n_mels=e["n_mels"], state=model.state_dict(),
                      mean=mean, std=std, conv_per_block=cfg["CONV_PER_BLOCK"])
        yt, yp = predict(model, frontend, mean, std, make_loader(all_idx[:400], 64, False, src=src))
        torch.save(ck, path)
        print(f"{exp}: xong sau {time.time() - t0:.0f}s | độ chính xác trên 400 clip đã học: {scores(yt, yp)[0]:.2%}")
        del model, frontend, out
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    bundles = []
    for exp in final_exps:
        ck = torch.load(final_path(exp), map_location="cpu", weights_only=False)
        bundles.append(dict(
            model=ck["model"], n_mels=ck["n_mels"], conv_per_block=ck.get("conv_per_block", 1),
            state={k: v.cpu() for k, v in ck["state"].items()},
            mean=float(ck["mean"]), std=float(ck["std"]),
            class_names=VI_LIST, group_names=GROUP_NAMES,
            sample_rate=ck.get("sample_rate", SR), n_fft=CFG["N_FFT"], hop_length=CFG["HOP_LENGTH"],
            duration=CFG["DURATION"], threshold=FINAL_THRESHOLD, experiment=f"{exp} (toàn bộ dữ liệu)",
        ))
    torch.save(dict(members=bundles, tta_shifts=list(TTA_SHIFTS)), DEMO_DIR / "demo_ensemble.pt")
    print("Đã lưu:", DEMO_DIR / "demo_ensemble.pt", "(dùng cho app.py)")

Các mô hình cuối: ['E9']


100%|██████████| 312M/312M [02:12<00:00, 2.47MB/s]


Đã nạp trọng số pretrain: 84 tensor


E9 | toàn bộ dữ liệu:   0%|          | 0/20 [00:00<?, ?it/s]

E9: xong sau 284s | độ chính xác trên 400 clip đã học: 98.75%
Đã lưu: /content/drive/MyDrive/ESC50/demo/demo_ensemble.pt (dùng cho app.py)


In [6]:
ck = torch.load(CKPT_DIR / f"{val_best}_fold{best_fold}.pt", map_location="cpu", weights_only=False)
torch.save(dict(model=ck["model"], n_mels=ck["n_mels"], conv_per_block=ck.get("conv_per_block", 1),
                state={k: v.cpu() for k, v in ck["state"].items()},
                mean=float(ck["mean"]), std=float(ck["std"]),
                class_names=VI_LIST, group_names=GROUP_NAMES,
                sample_rate=ck.get("sample_rate", SR), n_fft=CFG["N_FFT"], hop_length=CFG["HOP_LENGTH"],
                duration=CFG["DURATION"], threshold=VAL_THRESHOLD, experiment=val_best),
           DEMO_DIR / "demo_model.pt")
shutil.copy(DEMO_DIR / "demo_ensemble.pt", BEST_DIR / "best_ensemble_full.pt")
print("Đã lưu:", DEMO_DIR / "demo_model.pt", "và", DEMO_DIR / "demo_ensemble.pt")
print("Best model nằm ở:", BEST_DIR)

Đã lưu: /content/drive/MyDrive/ESC50/demo/demo_model.pt và /content/drive/MyDrive/ESC50/demo/demo_ensemble.pt
Best model nằm ở: /content/drive/MyDrive/ESC50/best_model
